[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/upm-classes/ai_medical_imaging_2026/blob/main/notebooks/student/NB07_transfer_learning.ipynb)

# NB07 · Transfer learning: scratch, frozen, fine-tuned
**Artificial Intelligence in Medical Imaging** · Day 3 · Lab 8 (33 min)

| | |
|---|---|
| Runtime | **GPU recommended** (Runtime → Change runtime type → T4). CPU works at 28 px with fewer epochs |
| Data | PneumoniaMNIST from MedMNIST+ at 64 px (28 px on CPU), CC BY 4.0 |
| Model | ResNet-18, ImageNet weights from torchvision |
| Checkpoints | output shape (batch, 1) · only the head trains in phase 1 · results table filled |
| Tiers | **Core** = scratch / frozen / fine-tuned at 100% of the labels · **Plus** = repeat at 10% · **Challenge** = 128 px and epochs to reach a target AUC |

**Learning goals.** (1) Adapt a pretrained ResNet to one-channel X-rays. (2) Run the two-phase fine-tuning recipe. (3) Compare scratch, frozen and fine-tuned models, with 100% and 10% of the labels, **against your prediction**. (4) Resist generalising from one run.

**Before you run anything:** fill in your prediction (the ranking you expect with 10% of the labels) in the cell below. Surprises are the lesson.

**Rules.** Choose on **validation**. The test set is evaluated **once**, at the end. Same seed for all runs.

In [ ]:
%pip install -q "medmnist==3.0.2"
import os, random, time, copy
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt, pandas as pd
import torch, torch.nn as nn, torchvision
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

def set_seed(seed=42):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
SEED = 42; set_seed(SEED); Path("data").mkdir(exist_ok=True); Path("outputs").mkdir(exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU = device.type == "cuda"
SIZE = 64 if GPU else 28                       # CPU fallback: 28 px and fewer epochs
EP = dict(scratch=8, frozen=8, phase1=3, phase2=5) if GPU else dict(scratch=4, frozen=4, phase1=2, phase2=2)
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| device", device, "| image size", SIZE, "| epochs", EP)
if not GPU: print("!! No GPU: using 28 px and few epochs. Results will be weaker; the lesson is the same. Runtime > Change runtime type > T4 GPU.")

In [ ]:
my_prediction = ""     # EDIT: e.g. "fine-tuned > frozen > scratch" for the 10% case; write it BEFORE you run the experiments
print("my prediction:", my_prediction or "(not written yet!)")

## 1. Data (MedMNIST+ at 64 px)

In [ ]:
def load_pneumonia(size):
    try:
        from medmnist import PneumoniaMNIST
        kw = {} if size == 28 else {"size": size}
        out = {}
        for s in ("train", "val", "test"):
            d = PneumoniaMNIST(split=s, download=True, root="data", **kw)
            out[s] = (d.imgs.reshape(len(d.imgs), size, size).astype(np.uint8), d.labels.reshape(-1).astype(np.int64))
        return out, True
    except Exception as e:
        print("!! Could not download PneumoniaMNIST:", repr(e), "-> SYNTHETIC data; numbers mean nothing")
        rng = np.random.default_rng(0); out = {}
        for s, n in (("train", 1500), ("val", 300), ("test", 300)):
            y = (rng.random(n) < 0.7).astype(np.int64); x = rng.normal(100, 25, (n, size, size)); x[y == 1, size // 4: size // 2, size // 4: size // 2] += 50
            out[s] = (x.clip(0, 255).astype(np.uint8), y)
        return out, False

data, REAL = load_pneumonia(SIZE)
to_t = lambda x: torch.from_numpy(x).float().div(255).unsqueeze(1)           # (N, 1, H, W) in 0..1: normalisation happens inside the model
X = {s: to_t(v[0]) for s, v in data.items()}; Y = {s: torch.from_numpy(v[1]) for s, v in data.items()}
print({s: tuple(X[s].shape) for s in X}, "| real:", REAL, "| pneumonia share in train:", round(float(Y["train"].float().mean()), 2))

def stratified_subset(y, frac, seed=SEED):
    """Indices of a class-balanced random subset with the same class proportions."""
    if frac >= 1.0: return np.arange(len(y))
    idx, _ = train_test_split(np.arange(len(y)), train_size=frac, stratify=y, random_state=seed)
    return np.sort(idx)

In [ ]:
sub = stratified_subset(Y["train"].numpy(), 0.10)
print("10% subset:", len(sub), "images | pneumonia share", round(float(Y["train"][sub].float().mean()), 3), "vs full", round(float(Y["train"].float().mean()), 3))
assert abs(len(sub) - 0.10 * len(Y["train"])) <= 1
assert abs(Y["train"][sub].float().mean() - Y["train"].float().mean()) < 0.01, "the subset must be stratified"
assert np.array_equal(sub, stratified_subset(Y["train"].numpy(), 0.10)), "same seed -> same subset"
print("subset checks passed")

## 2. The model: ResNet-18 adapted to X-rays
Three adaptations: (1) ImageNet networks expect **3 channels**: repeat the grey channel; (2) ImageNet **normalisation**; (3) a **new head**: one score per image (`Linear(512, 1)`, so the output shape is `(batch, 1)`).

**TODO 1.** Complete `forward` (repeat the channel 3 times) and `make_model` (replace `fc`). Hint: `x.repeat(1, 3, 1, 1)`.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

class XrayResNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        w = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None            # None = random initialisation
        self.net = resnet18(weights=w)
        # TODO 1a: replace the last layer by a new head: Linear(self.net.fc.in_features, 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, x):                      # x: (B, 1, H, W) in 0..1
        # TODO 1b: repeat the grey channel 3 times (hint: x.repeat(1, 3, 1, 1))
        return self.net((x - self.mean) / self.std)

def make_model(pretrained):
    try:
        return XrayResNet(pretrained)
    except Exception as e:
        print("!! Could not load ImageNet weights:", repr(e), "-> random initialisation; 'pretrained' results are NOT pretrained"); return XrayResNet(False)

In [ ]:
m = make_model(True)
out = m(X["val"][:4])
n_all = sum(p.numel() for p in m.parameters())
print("output shape:", tuple(out.shape), "| parameters:", f"{n_all:,}")
assert out.shape == (4, 1), "the model must output one score per image: (batch, 1)"
print("Checkpoint 1 passed")

## 3. Training helpers
`train_epochs` trains for a number of epochs and records the **validation AUC after each epoch**. Look at how the mode changes which parameters train:

| mode | what trains | BatchNorm layers |
|---|---|---|
| scratch | everything, random start | train mode |
| frozen | only the new head (`fc`) | **eval mode**: ImageNet statistics stay fixed |
| fine-tune, phase 1 | only the head | eval mode |
| fine-tune, phase 2 | everything, **smaller learning rate, new optimiser** | train mode |

> Creating a **new optimiser** after unfreezing is required: the old one only knows the head's parameters.

In [ ]:
def set_trainable(model, which):
    """which = 'all' or 'head'."""
    for p in model.parameters(): p.requires_grad = (which == "all")
    for p in model.net.fc.parameters(): p.requires_grad = True

def n_trainable(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)

@torch.no_grad()
def val_auc(model, split="val"):
    model.eval(); s = []
    for i in range(0, len(X[split]), 256): s.append(torch.sigmoid(model(X[split][i:i + 256].to(device))).cpu())
    return roc_auc_score(Y[split].numpy(), torch.cat(s).numpy().ravel())

def train_epochs(model, idx, epochs, lr, bn_eval=False, seed=SEED, curve=None, bs=64):
    """Train on the training images `idx` for `epochs` epochs with a NEW AdamW optimiser over the trainable parameters."""
    model.to(device); g = torch.Generator().manual_seed(seed)
    loader = DataLoader(TensorDataset(X["train"][idx], Y["train"][idx]), batch_size=bs, shuffle=True, generator=g)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    crit = nn.BCEWithLogitsLoss(); curve = [] if curve is None else curve
    for ep in range(epochs):
        model.train()
        if bn_eval:
            for mod in model.modules():
                if isinstance(mod, nn.BatchNorm2d): mod.eval()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.float().unsqueeze(1).to(device)
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        curve.append(val_auc(model))
    return curve

## 4. The three experiments
**TODO 2.** Complete `run(mode, frac)`:
- `scratch`: `make_model(False)`, train everything.
- `frozen`: `make_model(True)`, `set_trainable(m, "head")`, BatchNorm in eval mode.
- `finetune`: phase 1 as frozen (few epochs, `lr=1e-3`), then **phase 2**: `set_trainable(m, "all")`, `lr=1e-4`, a new optimiser (it is created inside `train_epochs`).

In [ ]:
def run(mode, frac=1.0, seed=SEED, ep=EP):
    set_seed(seed); idx = stratified_subset(Y["train"].numpy(), frac, seed); curve = []
    if mode == "scratch":
        # TODO 2a: random-init model, everything trainable, lr 1e-3
        m = None
    elif mode == "frozen":
        # TODO 2b: pretrained model, only the head trainable (set_trainable(m, "head")), BatchNorm in eval mode (bn_eval=True), lr 1e-3
        m = None
    elif mode == "finetune":
        # TODO 2c: phase 1 = as frozen, ep["phase1"] epochs;  phase 2 = set_trainable(m, "all"), ep["phase2"] epochs, lr 1e-4
        m = None
    else: raise ValueError(mode)
    return m, curve

### Checkpoint 2 · phase 1 trains only the head

In [ ]:
m_chk = make_model(True); set_trainable(m_chk, "head")
print("trainable parameters in phase 1:", n_trainable(m_chk), "of", sum(p.numel() for p in m_chk.parameters()))
assert n_trainable(m_chk) == 513, "only fc.weight (512) and fc.bias (1) should train in phase 1"
set_trainable(m_chk, "all"); assert n_trainable(m_chk) == sum(p.numel() for p in m_chk.parameters())
print("Checkpoint 2 passed")

### Core · all three at 100% of the labels
Run the three modes. Each run prints nothing until it finishes. Record the **final validation AUC** and the **epochs needed** to get within 0.01 of the best AUC of that run.

In [ ]:
results, curves, final_auc = [], {}, {}
def record(mode, frac):
    t0 = time.time(); m, c = run(mode, frac); curves[(mode, frac)] = c; final_auc[(mode, frac)] = c[-1]
    best = max(c); need = next(i + 1 for i, a in enumerate(c) if a >= best - 0.01)
    results.append(dict(model=mode, labels=f"{int(frac*100)}%", n_train=int(len(stratified_subset(Y["train"].numpy(), frac))), val_auc_final=round(c[-1], 3), epochs_to_best=need, minutes=round((time.time() - t0) / 60, 1)))
    print(results[-1]); return m

models = {}
for mode in ["scratch", "frozen", "finetune"]:
    models[(mode, 1.0)] = record(mode, 1.0)
table = pd.DataFrame(results); table

In [ ]:
assert len(table) == 3 and table["val_auc_final"].notna().all(), "fill the table: three runs at 100%"
assert (table["val_auc_final"] > 0.5).all(), "every model should beat chance on validation"
print("Checkpoint 3 passed: results table filled (100% of the labels)")

**Compare with your prediction.** Is the ordering what you expected? The validation set has 524 images, so one AUC has an uncertainty of about ±0.02 even if the ranking is perfect. Which differences exceed that?

### Plus · 10% of the labels (stratified, fixed seed)
Repeat with a class-balanced random 10% of the training images (about 470 images). Same seed, same epochs. **Post your 10% row on the board.**

In [ ]:
# [PLUS]
# TODO (Plus): call record(mode, 0.1) for scratch, frozen and finetune; then show the pivot table
...

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for k, frac in enumerate([1.0, 0.1]):
    for mode in ["scratch", "frozen", "finetune"]:
        if (mode, frac) in curves: ax[k].plot(range(1, len(curves[(mode, frac)]) + 1), curves[(mode, frac)], marker="o", label=mode)
    ax[k].set_title(f"{int(frac*100)}% of the labels"); ax[k].set_xlabel("epoch (fine-tune: phase 1 then phase 2)"); ax[k].set_ylabel("validation AUC"); ax[k].legend()
plt.tight_layout(); plt.show()

## 5. Report the test set once
Pick the model you would deploy **using validation only**, then evaluate it on the test set a single time (the helper counts calls). You may report all six if you like; that is still one evaluation per model, not a search over models.

In [ ]:
_test_calls = []
def test_once(model):
    """THE ONLY PLACE THAT TOUCHES THE TEST SET FOR A GIVEN MODEL. Call it once per model you report."""
    _test_calls.append(id(model)); 
    if _test_calls.count(id(model)) > 1: print("!! this model's test AUC was already computed: no second look")
    return val_auc(model, "test")

best_key = max((k for k in models if k[1] == 1.0), key=lambda k: final_auc[k])
print("chosen on validation (100% labels):", best_key)
test_auc = test_once(models[best_key])
print(f"test AUC of the chosen model: {test_auc:.3f}")
assert _test_calls.count(id(models[best_key])) == 1
print("Checkpoint 4 passed: the chosen model was evaluated on the test set once")

**Reflection questions.**
1. Which model needed the fewest epochs? Does that match Raghu et al. (2019), who found the clearest benefit of pretraining was faster convergence?
2. At 100% labels, is the gap between scratch and fine-tuned larger than the ±0.02 noise? At 10%?
3. ImageNet photos are in colour and show cats and cars. Why does it still help for X-rays? What could go wrong? (BatchNorm statistics, normalisation, resolution.)
4. One run, one seed, one dataset: what would you do before writing "pretraining helps" in a report?

### Challenge · 128 px and epochs to a target AUC
Load PneumoniaMNIST at `size=128` (MedMNIST+), repeat the *fine-tuned* and *scratch* runs at 10% of the labels, and plot **validation AUC against epoch**. Report the number of epochs each needs to reach AUC ≥ 0.90 (or the best of the two, if neither gets there). GPU strongly recommended.

In [ ]:
# [CHALLENGE]
# TODO (Challenge): load size=128 data (load_pneumonia(128)), swap the global X, Y, run scratch and finetune at 10%,
# restore X, Y afterwards, and plot validation AUC against epoch; report epochs to reach AUC 0.90.
...

## Summary
- A pretrained ResNet needs three adaptations for X-rays: 3 channels, ImageNet normalisation, a new head.
- Two-phase recipe: head first (frozen backbone, BatchNorm in eval), then everything with a smaller learning rate and a **new optimiser**.
- Pretraining mostly buys speed and robustness with few labels; the gap at 100% labels may be small. One run is not a law.
- Choose on validation; report the test set once.

*Next:* NB08 (segmentation with a U-Net).